# Stage 9B.0 - Nominal F300 4F Virtual Bench and Candidate Atlas

This notebook is a nominal, unvalidated scalar 4F diagnostic only. It does not make the physical 4F route ready, does not model a camera, does not run inverse correction or AI, and does not introduce material response.

Boundary labels: `nominal_4f_forward_model`, `not_bench_calibrated`, `not_physical_4f_readiness_ready`, `not_camera_modelled`, `not_material_modelled`, `final_export_allowed=False`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from vbb_study.digital_twin.nominal_f300_4f import (
    CLAIM_BOUNDARY_LABELS,
    NominalF300Config,
    config_from_profile,
    load_nominal_f300_profile,
    nominal_4f_sanity_report,
    plot_component_sequence,
    run_nominal_f300_4f,
    run_to_manifest,
)
from vbb_study.digital_twin.candidate_beam_atlas import (
    DEFAULT_RUN_ID,
    build_candidate_specs,
    export_candidate_package,
    plot_candidate_atlas,
    plot_stop_robustness,
    relay_output_candidate_metrics,
    simulate_candidate,
)


In [ ]:
profile = load_nominal_f300_profile()
study = json.loads(Path('configs/studies/cslm_nominal_4f_candidate_atlas_v1.json').read_text())
display(Markdown('## Nominal Profile'))
display(pd.DataFrame([
    {'parameter': key, 'value': entry['value'], 'provenance': entry['provenance']}
    for key, entry in profile['known_nominal_geometry'].items()
]))
display(Markdown('## Boundary'))
display(pd.Series({label: True for label in CLAIM_BOUNDARY_LABELS + ('final_export_allowed_false',)}))


In [ ]:
config = config_from_profile()
run = run_nominal_f300_4f(config)
manifest = run_to_manifest(run)
display(Markdown('## Executed Nominal Component Chain'))
display(pd.DataFrame(manifest['component_manifest']))
display(Markdown('## Energy Ledger'))
display(pd.DataFrame(manifest['component_energy_ledger']))
display(Markdown('## Sanity Report'))
display(pd.Series(nominal_4f_sanity_report(run)))


In [ ]:
component_fig = plot_component_sequence(run)
stop_fig = plot_stop_robustness()
atlas_fig = plot_candidate_atlas()
for fig in (component_fig, stop_fig, atlas_fig):
    display(Image(filename=str(fig)))


In [ ]:
specs = build_candidate_specs()
rows = []
for spec in specs:
    candidate_run = simulate_candidate(spec, NominalF300Config.fast())
    rows.append({'candidate_id': spec.candidate_id, **relay_output_candidate_metrics(candidate_run)})
display(pd.DataFrame(rows))


In [ ]:
package_spec = [spec for spec in build_candidate_specs() if spec.candidate_id == 'vortex_ell_2'][0]
package_paths = export_candidate_package(package_spec, run_id=DEFAULT_RUN_ID)
display(Markdown('## Demonstrator Candidate Package'))
display(pd.Series({key: str(path) for key, path in package_paths.items()}))
claim_boundary_path = Path('outputs/nominal_4f_candidate_runs') / DEFAULT_RUN_ID / package_spec.candidate_id / 'claim_boundary.md'
display(Markdown(claim_boundary_path.read_text(encoding='utf-8')))


## Unsupported

Still unsupported: measured physical 4F coordinates, true stop radius/position, SLM phase response calibration, camera modelling, inverse correction, AI, relay-output-to-axicon handoff geometry, material response, and final export.